In [7]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import signal
from scipy.fft import fft
import warnings
warnings.filterwarnings('ignore')


In [8]:
preprocessed_dir = Path ("/Users/folasewaabdulsalam/Downloads/EEG_Based_Mental_Workload_Classifier/EEG-Based_Mental_Workload_Classifier/preprocessed_data")

channel_names = ['AF3', 'F7', 'F3', 'FC5', 'T7', 'P7', 'O1', 'O2', 'P8', 'T8', 'FC6', 'F4', 'F8', 'AF4']

sampling_frequency = 128
preprocessed_files = sorted(preprocessed_dir.glob('*.npy'))
print(f"Found {len(preprocessed_files)} preprocessed files")

sample_data = np.load(preprocessed_files[0])
print(f"Sample Shape is {sample_data.shape}")


Found 93 preprocessed files
Sample Shape is (19200, 14)


In [9]:
#define frequency bands
frequency_bands = {'delta': (0.5,4),
                   'theta': (4,8),
                   'alpha': (8,13),
                   'beta': (13, 30),
                   }
for band, (low,high) in frequency_bands.items():
    print(f"{band.capitalize()}:{low}-{high}Hz")



Delta:0.5-4Hz
Theta:4-8Hz
Alpha:8-13Hz
Beta:13-30Hz


In [10]:
#extract psd features from one window

def extract_psd_features(window, sampling_frequency, frequency_bands):

    n_channels = window.shape[1]
    features = []

    #for each channel
    for ch in range(n_channels):
        freqs, psd = signal.welch(window[:,ch], fs=sampling_frequency, nperseg=256) #psd using welch

        for band_name, (low_freq, high_freq) in frequency_bands.items():
            idx_band = np.logical_and(freqs>=low_freq, freqs<=high_freq)
            band_power = np.mean(psd[idx_band])
            features.append(band_power)

    return np.array(features)
print("PSD Extraction Done!")


PSD Extraction Done!


In [11]:
#Creating a sliding window function of size 512 and shift 128 
def create_sliding_window(data, window_size=512, shift=128):

    n_samples = data.shape[0]
    windows = []

    #sliding window across the data
    start = 0
    while start+window_size <=n_samples:
        window = data[start:start + window_size, :]
        windows.append(window)
        start+=shift
    return windows
print("Sliding window done!")



Sliding window done!


In [12]:
#extract features from one recording
def extract_features_from_recording(data, sampling_frequency, frequency_bands, window_size=512, shift=128):

    #create sliding windows
    windows = create_sliding_window(data, window_size, shift)

    #extract psd features from each window
    window_features = []
    for window in windows:
        features = extract_psd_features(window, sampling_frequency, frequency_bands)
        window_features.append(features)
    
    #averaging the features across windows
    avg_features = np.mean(window_features, axis=0)

    return avg_features
print("Feature Extraction Done!")


Feature Extraction Done!


In [14]:
#looping through all the preprocessed files to extract features

all_features = []
all_labels = []
all_subjects = []

for i, file_path in enumerate(preprocessed_files):
    print(f"Preprocessing {i+1}/93: {file_path.name}...", end='\r')

    data = np.load(file_path)
    filename = file_path.stem
    parts = filename.split('_')
    subject = parts[0]
    workload = parts[1]

    #extract features
    features = extract_features_from_recording(data, sampling_frequency, frequency_bands, window_size=512, shift=128)

    all_features.append(features)
    all_labels.append(workload)
    all_subjects.append(subject)
print("\n Feature extraction complete!")
print(f"   Total recordings: {len(all_features)}")
print(f"   Features per recording: {len(all_features[0])}")




Preprocessing 93/93: sub48_lo.npy...
 Feature extraction complete!
   Total recordings: 93
   Features per recording: 56


In [16]:
#save the features to a dataframe
features_df = pd.DataFrame(all_features)
features_df['subject'] = all_subjects
features_df['workload'] = all_labels

#creating feature column names
feature_names = []
for band in ['delta', 'theta', 'alpha', 'beta']:
    for channel in channel_names:
        feature_names.append(f"{band}_{channel}")

#rename column
features_df.columns = feature_names + ['subject', 'workload']
features_df.to_csv('/Users/folasewaabdulsalam/Downloads/EEG_Based_Mental_Workload_Classifier/EEG-Based_Mental_Workload_Classifier/features.csv', index=False)

print(f"Features DataFrame shape: {features_df.shape}")
print(features_df.head())

Features DataFrame shape: (93, 58)
     delta_AF3    delta_F7    delta_F3  delta_FC5     delta_T7    delta_P7  \
0    10.725343    5.077131    2.238050   0.697380    56.521224   15.568467   
1  5516.791766  963.839271  128.723981  12.964473  1396.342661  220.522152   
2     3.989949    2.608310    1.053837   0.753365    19.472834   10.321517   
3     2.253481    0.975757    0.694703   0.259047     2.421729    0.950285   
4    55.389697    9.962111    4.743549   0.646834   103.442932   12.132124   

    delta_O1  delta_O2     delta_P8    delta_T8  ...      beta_O1     beta_O2  \
0   3.971748  1.247739     5.578479    1.934691  ...    30.620624    9.059456   
1  44.662870  6.670238  5436.258042  945.965484  ...  5296.014472  923.484204   
2   4.888012  3.832464     4.447023    2.039295  ...    18.877661    9.902051   
3   0.583619  0.401876     8.895102    1.115093  ...     4.412904    1.214817   
4   5.344133  0.949898     8.043204    3.059159  ...    87.510300   12.516137   

      bet